# Trial Analytics — `logs/exp1`

Loads every TensorBoard run under `logs/exp1/`, groups experiments by base name (strips `_t1`, `_t2`, … trial suffixes), and plots:

- Episode reward
- Episode length
- Overall win rate
- Per-opponent win rates (auto-detected)
- Final win-rate summary table

**Kernel**: this notebook requires the `env_sfrl` conda environment.  
Register it once if needed:
```bash
conda activate env_sfrl
python -m ipykernel install --user --name env_sfrl --display-name "env_sfrl"
```
Then select **env_sfrl** from the kernel menu and re-run.

In [ ]:
import os, re, warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")

try:
    from tensorboard.backend.event_processing import event_accumulator
except ImportError:
    raise ImportError(
        "tensorboard not found.\n"
        "Register env_sfrl as a kernel:\n"
        "  conda activate env_sfrl && "
        "python -m ipykernel install --user --name env_sfrl --display-name env_sfrl\n"
        "Then switch to that kernel and re-run."
    )

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# ── Configuration ────────────────────────────────────────────────────────────
EXP_DIR        = Path("logs/exp1")
SMOOTH         = 7      # rolling-mean window; set to 1 to disable smoothing
TAIL_FRAC      = 0.10   # fraction of steps used for the final summary table

# Add group names here to hide them from all plots (e.g. housekeeping runs):
EXCLUDE_GROUPS = set()  # e.g. {"merged", "to_be_merged"}

In [ ]:
# ── Data Discovery & Loading ─────────────────────────────────────────────────
TRIAL_RE = re.compile(r"_t(\d+)$")

SCALAR_TAGS = [
    "rollout/ep_rew_mean",
    "rollout/ep_len_mean",
    "rollout/ep_winrate",
    "rollout/ep_winrate_ryu",
    "rollout/ep_winrate_ken",
    "rollout/ep_winrate_chunli",
    "rollout/ep_winrate_guile",
]


def parse_run_info(run_dir: Path, base: Path):
    """Return (group, trial_num) for a run directory."""
    rel    = run_dir.relative_to(base)
    parts  = list(rel.parts)
    leaf   = parts[-1]
    m      = TRIAL_RE.search(leaf)
    trial  = int(m.group(1)) if m else 1
    stem   = leaf[:m.start()] if m else leaf
    parent = "/".join(parts[:-1])
    group  = f"{parent}/{stem}" if parent else stem
    return group, trial


def load_scalars(run_dir: Path) -> dict:
    ea = event_accumulator.EventAccumulator(
        str(run_dir), size_guidance={event_accumulator.SCALARS: 0}
    )
    ea.Reload()
    available = set(ea.Tags().get("scalars", []))
    out = {}
    for tag in SCALAR_TAGS:
        if tag in available:
            events = ea.Scalars(tag)
            out[tag] = pd.DataFrame(
                [(e.step, e.value) for e in events], columns=["step", "value"]
            )
    return out


# Discover all leaf directories that contain tfevents files
all_data: dict[str, dict[int, dict]] = defaultdict(dict)

for dirpath, _, files in os.walk(EXP_DIR):
    if not any("tfevents" in f for f in files):
        continue
    run_dir = Path(dirpath)
    group, trial = parse_run_info(run_dir, EXP_DIR)
    if group in EXCLUDE_GROUPS:
        continue
    print(f"  {group:45s} trial {trial}")
    all_data[group][trial] = load_scalars(run_dir)

print(f"\n✓ {len(all_data)} experiment groups loaded.")

In [ ]:
# ── Experiment Summary ───────────────────────────────────────────────────────
rows = []
for group, trials in sorted(all_data.items()):
    for trial_num, scalars in sorted(trials.items()):
        rew_df = scalars.get("rollout/ep_rew_mean")
        steps  = (
            f"{rew_df['step'].min():,} – {rew_df['step'].max():,}"
            if rew_df is not None else "—"
        )
        opps = sorted(
            tag.split("rollout/ep_winrate_")[-1]
            for tag in scalars
            if tag.startswith("rollout/ep_winrate_")
        )
        rows.append({
            "Group":      group,
            "Trial":      trial_num,
            "Step range": steps,
            "Opponents":  ", ".join(opps) or "—",
        })

pd.set_option("display.max_rows", 80)
pd.DataFrame(rows)

In [ ]:
# ── Shared Plotting Utilities ────────────────────────────────────────────────
groups_sorted = sorted(all_data.keys())

# Build a stable color map (handles up to 40 groups)
_pal = np.vstack([
    plt.cm.tab20(np.linspace(0, 1, 20)),
    plt.cm.tab20b(np.linspace(0, 1, 20)),
])
COLOR    = {g: _pal[i % len(_pal)] for i, g in enumerate(groups_sorted)}
LS_CYCLE = ["-", "--", "-.", ":"]


def smooth_s(df: pd.DataFrame, w: int) -> pd.DataFrame:
    if w <= 1:
        return df
    return df.assign(value=df["value"].rolling(w, min_periods=1, center=True).mean())


def plot_metric(tag: str, ylabel: str, title: str,
                pct_axis=False, figsize=(15, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    plotted = False
    for group in groups_sorted:
        color  = COLOR[group]
        trials = all_data[group]
        n      = len(trials)
        for i, (trial_num, scalars) in enumerate(sorted(trials.items())):
            if tag not in scalars:
                continue
            df  = smooth_s(scalars[tag], SMOOTH)
            lbl = group if n == 1 else f"{group} t{trial_num}"
            ax.plot(df["step"], df["value"],
                    color=color, linestyle=LS_CYCLE[i % 4],
                    linewidth=1.5, alpha=0.88, label=lbl)
            plotted = True
    if not plotted:
        ax.text(0.5, 0.5, f"No data for: {tag}",
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
    if pct_axis:
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.set_xlabel("Timestep")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold", fontsize=12)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, fontsize=7, ncol=3,
              loc="lower right", framealpha=0.6)
    plt.tight_layout()
    plt.show()

## Episode Reward

In [ ]:
plot_metric(
    "rollout/ep_rew_mean",
    "Mean Episode Reward",
    "Episode Reward over Time",
)

## Episode Length

In [ ]:
plot_metric(
    "rollout/ep_len_mean",
    "Mean Episode Length (steps)",
    "Episode Length over Time",
)

## Overall Win Rate

In [ ]:
plot_metric(
    "rollout/ep_winrate",
    "Win Rate",
    "Overall Win Rate over Time",
    pct_axis=True,
)

## Per-Opponent Win Rates

In [ ]:
# Auto-detect which opponents appear in the loaded data
all_opps = sorted({
    tag.split("rollout/ep_winrate_")[-1]
    for trials in all_data.values()
    for scalars in trials.values()
    for tag in scalars
    if tag.startswith("rollout/ep_winrate_")
})

if not all_opps:
    print("No per-opponent win rate data found.")
else:
    ncols  = len(all_opps)
    fig, axes = plt.subplots(1, ncols,
                              figsize=(6 * ncols, 5),
                              sharey=True)
    if ncols == 1:
        axes = [axes]

    for ax, opp in zip(axes, all_opps):
        tag = f"rollout/ep_winrate_{opp}"
        for group in groups_sorted:
            color  = COLOR[group]
            trials = all_data[group]
            n      = len(trials)
            for i, (trial_num, scalars) in enumerate(sorted(trials.items())):
                if tag not in scalars:
                    continue
                df  = smooth_s(scalars[tag], SMOOTH)
                lbl = group if n == 1 else f"{group} t{trial_num}"
                ax.plot(df["step"], df["value"],
                        color=color, linestyle=LS_CYCLE[i % 4],
                        linewidth=1.5, alpha=0.88, label=lbl)

        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        ax.set_xlabel("Timestep")
        if ax is axes[0]:
            ax.set_ylabel("Win Rate")
        ax.set_title(f"vs {opp.capitalize()}", fontweight="bold")
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, labels, fontsize=7, ncol=2,
                  loc="lower right", framealpha=0.6)

    fig.suptitle("Per-Opponent Win Rates over Time",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Final Win-Rate Summary

Average win rates over the **last `TAIL_FRAC` fraction** of each run's timesteps.

In [ ]:
rows = []
for group in groups_sorted:
    for trial_num, scalars in sorted(all_data[group].items()):
        wr_df = scalars.get("rollout/ep_winrate")
        if wr_df is None:
            continue
        cutoff  = wr_df["step"].max() * (1 - TAIL_FRAC)
        tail_wr = wr_df[wr_df["step"] >= cutoff]["value"]
        row = {
            "Group":       group,
            "Trial":       trial_num,
            "Overall WR":  f"{tail_wr.mean():.1%}" if len(tail_wr) else "—",
        }
        for opp in all_opps:
            odf = scalars.get(f"rollout/ep_winrate_{opp}")
            if odf is not None:
                c     = odf["step"].max() * (1 - TAIL_FRAC)
                tail  = odf[odf["step"] >= c]["value"]
                row[f"vs {opp.capitalize()}"] = (
                    f"{tail.mean():.1%}" if len(tail) else "—"
                )
            else:
                row[f"vs {opp.capitalize()}"] = "—"
        rows.append(row)

summary = pd.DataFrame(rows)
summary